# Tinker -> vLLM adapter conversion (ported from huikang/tinker-submission-notebook)

Tinker's raw adapter doesn't load in vLLM for two structural reasons:
1. Expert LoRAs are stored as a single 3D tensor at `.experts.w1` / `.experts.w2`
   with leading dim = num_experts; vLLM expects per-expert keys
   `.experts.{i}.up_proj` / `.experts.{i}.down_proj`.
2. Nemotron's Mamba block fuses `gate_proj` and `x_proj` into a single
   `in_proj` at serving time, but Tinker trains the two LoRAs separately
   (rank 32 each = effective rank 64). We compress to rank 32 via QR+SVD.

Source: https://www.kaggle.com/code/huikang/tinker-submission-notebook

In [ ]:
import os, glob, json, shutil
import re, torch
from safetensors import safe_open
from safetensors.torch import save_file

src_dir = os.path.dirname(next(iter(
    glob.glob('/kaggle/input/**/adapter_config.json', recursive=True))))
print('Tinker raw adapter dir:', src_dir)
for f in sorted(os.listdir(src_dir)):
    print(' ', f, os.path.getsize(os.path.join(src_dir, f)))
base_model = next(iter(glob.glob(
    '/kaggle/input/**/nemotron-3-nano*30b*/transformers/default/1', recursive=True)))
print('\nBase model dir:', base_model)

In [ ]:
# Stage everything in /kaggle/working so we can edit in place
os.makedirs('/kaggle/working/adapter', exist_ok=True)
for f in ('adapter_config.json', 'adapter_model.safetensors'):
    shutil.copy(os.path.join(src_dir, f), f'/kaggle/working/adapter/{f}')
os.chdir('/kaggle/working/adapter')
print('Staged:', os.listdir('.'))

In [ ]:
# Cell A: rewrite adapter_config.json to the form vLLM/PEFT expects on Nemotron.
with open('adapter_config.json') as f:
    cfg = json.load(f)
cfg['target_modules'] = ['k_proj','o_proj','in_proj','q_proj','up_proj','v_proj','down_proj','out_proj','lm_head']
# Tinker uses rank_pattern={'in_proj':64}, but after SVD fusion we collapse to rank 32.
cfg['rank_pattern'] = {}
cfg['alpha_pattern'] = {}
cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
cfg['inference_mode'] = True
with open('adapter_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print(json.dumps(cfg, indent=2))

In [ ]:
# Cell B: load the Nemotron base-model safetensors keys so we can look up the
# expected in_proj dimension for the SVD fusion step.
model_keys = set()
for st in glob.glob(f'{base_model}/*.safetensors'):
    with safe_open(st, framework='pt', device='cpu') as f:
        for key in f.keys():
            ts = f.get_slice(key)
            model_keys.add((key, tuple(ts.get_shape()), ts.get_dtype()))
model_key_shapes = {k: s for k, s, _ in model_keys}
print('Base-model keys loaded:', len(model_keys))

In [ ]:
# Cell C: the actual conversion (verbatim ported from huikang/tinker-submission-notebook cell 24).
def trained_adapter_key_rename(key_name):
    return key_name.replace('base_model.model.model', 'base_model.model.backbone')

adapter_tensors = {}
with safe_open('adapter_model.safetensors', framework='pt', device='cpu') as f:
    for key in f.keys():
        adapter_tensors[key] = f.get_tensor(key)
print('Loaded', len(adapter_tensors), 'trained tensors')

base_names = set()
for key in adapter_tensors:
    base = re.sub(r'\.lora_[AB]\.weight$', '', key)
    base_names.add(base)

mamba_merge_layers = {}
for base in base_names:
    for proj in ('gate_proj', 'x_proj'):
        if f'.{proj}' in base:
            layer_path = base.rsplit(f'.{proj}', 1)[0]
            mamba_merge_layers.setdefault(layer_path, {})[proj] = base
mamba_merge_bases = set()
for projs in mamba_merge_layers.values():
    mamba_merge_bases.update(projs.values())

tensors = {}
for base in sorted(base_names):
    lora_A = adapter_tensors[f'{base}.lora_A.weight']
    lora_B = adapter_tensors[f'{base}.lora_B.weight']
    renamed = trained_adapter_key_rename(base)
    if '.experts.w3' in base and lora_A.numel() == 0:
        continue
    if base in mamba_merge_bases:
        continue
    if '.experts.w1' in base or '.experts.w2' in base:
        if lora_A.shape[0] == 1:
            lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
        elif lora_B.shape[0] == 1:
            lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()
        num_experts = lora_A.shape[0]
        proj_name = 'up_proj' if '.w1' in base else 'down_proj'
        for i in range(num_experts):
            exp_renamed = re.sub(r'\.experts\.w[12]', f'.experts.{i}.{proj_name}', renamed)
            tensors[f'{exp_renamed}.lora_A.weight'] = lora_A[i].contiguous()
            tensors[f'{exp_renamed}.lora_B.weight'] = lora_B[i].contiguous()
        continue
    tensors[f'{renamed}.lora_A.weight'] = lora_A
    tensors[f'{renamed}.lora_B.weight'] = lora_B

for layer_path, projs in sorted(mamba_merge_layers.items()):
    renamed_layer = trained_adapter_key_rename(layer_path)
    in_proj_base = f'{renamed_layer}.in_proj'
    model_in_proj_key = renamed_layer.replace('base_model.model.', '') + '.in_proj.weight'
    in_proj_dim = model_key_shapes[model_in_proj_key][0]
    gate_A = adapter_tensors[f"{projs['gate_proj']}.lora_A.weight"].float()
    gate_B = adapter_tensors[f"{projs['gate_proj']}.lora_B.weight"].float()
    x_A    = adapter_tensors[f"{projs['x_proj']}.lora_A.weight"].float()
    x_B    = adapter_tensors[f"{projs['x_proj']}.lora_B.weight"].float()
    rank = gate_A.shape[0]
    A_cat = torch.cat([gate_A, x_A], dim=0)
    B_block = torch.zeros(in_proj_dim, 2 * rank)
    B_block[:gate_B.shape[0], :rank] = gate_B
    B_block[gate_B.shape[0]:gate_B.shape[0]+x_B.shape[0], rank:] = x_B
    Q_B, R_B = torch.linalg.qr(B_block)
    Q_A, R_A = torch.linalg.qr(A_cat.T)
    core = R_B @ R_A.T
    U, S, Vh = torch.linalg.svd(core, full_matrices=False)
    k = rank
    new_B = (Q_B @ U[:, :k]) * S[:k].unsqueeze(0)
    new_A = Vh[:k, :] @ Q_A.T
    kept = S[:k].sum().item(); total = S.sum().item()
    print(f'{layer_path}: SVD kept {kept:.2f}/{total:.2f} ({kept/total*100:.1f}%)')
    tensors[f'{in_proj_base}.lora_A.weight'] = new_A
    tensors[f'{in_proj_base}.lora_B.weight'] = new_B

print(f'\nConverted {len(adapter_tensors)} -> {len(tensors)} output tensors')
save_file(tensors, 'adapter_model.safetensors')

In [ ]:
# Cell D: package as submission.zip
import zipfile
out = '/kaggle/working/submission.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ('adapter_config.json', 'adapter_model.safetensors'):
        zf.write(fname, arcname=fname)
print('Wrote', out, '=', os.path.getsize(out), 'bytes')